# Notebook 04: Sheet Dynamics and Constraint Relaxation — Full Upgrade

This notebook upgrades the pseudo-time current-sheet dynamics into a more analysis-ready workflow.

It includes:

- a fragmented initial current sheet
- pseudo-time evolution with diffusion, restoring drift, and activity-driven forcing
- 45° contour tracking on cross-sheet cosine maps
- threshold masks and central-band masks
- time-series metrics for pocket count, width variation, and below-threshold fraction
- a **constraint-energy** order parameter
- a **critical-step proxy** based on the strongest growth in below-threshold fraction
- a **two-regime comparison** to show how parameter choices affect relaxation

This is still **not** a full resistive or kinetic reconnection model. It is a controlled geometric dynamics model.

Core gate:

\[
\cos\theta \ge \frac{1}{\sqrt{1^2 + 1^2}}
\]

Interpretation:

- above threshold: broadly aligned field
- below threshold: strong-shear region
- evolving geometry: a bounded fragmentation-and-relaxation process

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

np.random.seed(21)

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 360, 280
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

dx = x[1] - x[0]
dy = y[1] - y[0]

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)
print("dx, dy =", dx, dy)

## 1. Helpers

These are the shared geometry, measurement, and update functions.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def harris_target(X, Y, delta=0.35):
    return np.tanh(Y / delta)

def initial_field(X, Y, delta=0.35, base_eps=0.06, frag_amp=0.28, sigma=0.38):
    Bx = np.tanh(Y / delta)

    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = [0.0, 0.8, 1.7, 2.6]
    amps = [frag_amp, 0.75 * frag_amp, 0.55 * frag_amp, 0.35 * frag_amp]
    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

    By = By_background + By_fragment
    return Bx, By

def laplacian(F, dx, dy):
    return (
        (np.roll(F, 1, axis=0) - 2 * F + np.roll(F, -1, axis=0)) / dy**2 +
        (np.roll(F, 1, axis=1) - 2 * F + np.roll(F, -1, axis=1)) / dx**2
    )

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def failed_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def activity_map(cos_map, threshold=THRESHOLD):
    act = np.maximum(0.0, threshold - cos_map)
    return np.nan_to_num(act, nan=0.0)

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    rows, cols = mask.shape
    count = 0
    sizes = []

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] and not visited[i, j]:
                count += 1
                q = deque([(i, j)])
                visited[i, j] = True
                size = 0

                while q:
                    r, c = q.popleft()
                    size += 1
                    for rr, cc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                        if 0 <= rr < rows and 0 <= cc < cols:
                            if mask[rr, cc] and not visited[rr, cc]:
                                visited[rr, cc] = True
                                q.append((rr, cc))
                sizes.append(size)
    return count, sizes

def estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD):
    widths = np.full(cos_map.shape[1], np.nan)
    for j in range(cos_map.shape[1]):
        col = cos_map[:, j]
        bad = np.isfinite(col) & (col < threshold)
        if np.any(bad):
            y_bad = y[bad]
            widths[j] = y_bad.max() - y_bad.min()
    return widths

def local_min_count_1d(arr):
    arr = np.asarray(arr, dtype=float)
    good = np.isfinite(arr)
    idx = np.where(good)[0]
    if len(idx) < 3:
        return 0
    vals = arr[idx]
    count = 0
    for k in range(1, len(vals) - 1):
        if vals[k] < vals[k - 1] and vals[k] < vals[k + 1]:
            count += 1
    return count

def constraint_energy(cos_map, threshold=THRESHOLD):
    return float(np.nanmean(np.maximum(0.0, threshold - cos_map)))

def correlation_length_proxy(mask):
    # Mean contiguous True-run length across rows, ignoring empty rows.
    runs = []
    arr = np.nan_to_num(mask, nan=False).astype(bool)
    for row in arr:
        length = 0
        local_runs = []
        for val in row:
            if val:
                length += 1
            elif length > 0:
                local_runs.append(length)
                length = 0
        if length > 0:
            local_runs.append(length)
        runs.extend(local_runs)
    if len(runs) == 0:
        return 0.0
    return float(np.mean(runs))

def one_step(Bx, By, Bx_target, seed_pattern, dt, dx, dy, nu, alpha, beta, gamma):
    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    act = activity_map(cos_map, threshold=THRESHOLD)

    Bx_new = Bx + dt * (
        nu * laplacian(Bx, dx, dy)
        - alpha * (Bx - Bx_target)
    )

    By_new = By + dt * (
        nu * laplacian(By, dx, dy)
        + beta * act * seed_pattern
        - gamma * By
    )

    # Soft boundary restoration at top and bottom
    Bx_new[0, :] = Bx_target[0, :]
    Bx_new[-1, :] = Bx_target[-1, :]
    By_new[0, :] = 0.0
    By_new[-1, :] = 0.0

    return Bx_new, By_new

## 2. Initial fragmented field

We start from the same moderately fragmented regime used previously.

In [ ]:
delta = 0.35
frag_amp = 0.28
sigma_seed = 0.38

Bx0, By0 = initial_field(X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp, sigma=sigma_seed)
Bx_target = harris_target(X, Y, delta=delta)

Bx_hat0, By_hat0 = normalize_field(Bx0, By0)
cos0 = cross_sheet_cosine(Bx_hat0, By_hat0, shift=3)
mask0 = failed_mask(cos0)

print("initial below-threshold fraction =", np.nanmean(mask0))
print("initial constraint energy =", constraint_energy(cos0))

In [ ]:
skip = 12
plt.figure(figsize=(8, 5))
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip], Bx_hat0[::skip, ::skip], By_hat0[::skip, ::skip])
plt.xlabel("x")
plt.ylabel("y")
plt.title("Initial Normalized Magnetic Field")
plt.show()

In [ ]:
plt.figure(figsize=(9, 4.8))
plt.imshow(cos0, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", aspect="auto")
plt.colorbar(label="cross-sheet cosine")
plt.contour(X, Y, cos0, levels=[THRESHOLD], colors="white", linewidths=1.8, linestyles="--")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Initial Cross-Sheet Cosine Map")
plt.show()

## 3. Dynamics parameters

We define a baseline regime, then later compare it with a second regime.

In [ ]:
base_params = {
    "dt": 0.0018,
    "nu": 0.0012,
    "alpha": 0.25,
    "beta": 0.85,
    "gamma": 0.18,
    "n_steps": 220,
}

save_steps = [0, 20, 50, 100, 160, 220]

ks = [1.0, 2.0, 3.6, 5.0]
phases = [0.0, 0.8, 1.7, 2.6]
amps = [frag_amp, 0.75 * frag_amp, 0.55 * frag_amp, 0.35 * frag_amp]
seed_pattern = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma_seed)

base_params

## 4. Simulation runner

This wrapper computes snapshots and all tracked metrics in one place.

In [ ]:
def run_simulation(Bx_init, By_init, Bx_target, seed_pattern, params, save_steps):
    dt = params["dt"]
    nu = params["nu"]
    alpha = params["alpha"]
    beta = params["beta"]
    gamma = params["gamma"]
    n_steps = params["n_steps"]

    snapshots = {}
    metrics = {
        "step": [],
        "failed_fraction": [],
        "central_failed_fraction": [],
        "component_count": [],
        "width_mean": [],
        "width_std": [],
        "centerline_min": [],
        "centerline_min_count": [],
        "constraint_energy": [],
        "corr_length_proxy": [],
    }

    Bx_t = Bx_init.copy()
    By_t = By_init.copy()

    for step in range(n_steps + 1):
        Bx_hat_t, By_hat_t = normalize_field(Bx_t, By_t)
        cos_map_t = cross_sheet_cosine(Bx_hat_t, By_hat_t, shift=3)
        mask_t = failed_mask(cos_map_t)
        central_t = central_band_mask(mask_t, Y, half_width=0.8)

        comp_count, comp_sizes = connected_components(np.nan_to_num(central_t, nan=False).astype(bool))
        widths_t = estimate_failed_width_vs_x(cos_map_t, y, threshold=THRESHOLD)

        center_row = np.argmin(np.abs(y - 0.0))
        centerline_t = cos_map_t[center_row, :]

        metrics["step"].append(step)
        metrics["failed_fraction"].append(float(np.nanmean(mask_t)))
        metrics["central_failed_fraction"].append(float(np.nanmean(central_t)))
        metrics["component_count"].append(int(comp_count))
        metrics["width_mean"].append(float(np.nanmean(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
        metrics["width_std"].append(float(np.nanstd(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
        metrics["centerline_min"].append(float(np.nanmin(centerline_t)))
        metrics["centerline_min_count"].append(int(local_min_count_1d(centerline_t)))
        metrics["constraint_energy"].append(constraint_energy(cos_map_t))
        metrics["corr_length_proxy"].append(correlation_length_proxy(central_t))

        if step in save_steps:
            snapshots[step] = {
                "Bx": Bx_t.copy(),
                "By": By_t.copy(),
                "cos_map": cos_map_t.copy(),
                "mask": mask_t.copy(),
                "central": central_t.copy(),
                "centerline": centerline_t.copy(),
                "widths": widths_t.copy(),
            }

        if step < n_steps:
            Bx_t, By_t = one_step(
                Bx_t, By_t, Bx_target, seed_pattern,
                dt=dt, dx=dx, dy=dy,
                nu=nu, alpha=alpha, beta=beta, gamma=gamma
            )

    metrics["critical_step_proxy"] = int(np.argmax(np.gradient(metrics["failed_fraction"])))
    return snapshots, metrics

## 5. Baseline evolution

In [ ]:
snapshots_base, metrics_base = run_simulation(Bx0, By0, Bx_target, seed_pattern, base_params, save_steps)

print("baseline critical-step proxy =", metrics_base["critical_step_proxy"])
print("saved snapshots:", list(snapshots_base.keys()))

## 6. Baseline snapshot cosine maps

In [ ]:
for step in save_steps:
    snap = snapshots_base[step]
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        snap["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="cross-sheet cosine")
    plt.contour(
        X, Y, snap["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.8,
        linestyles="--"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Baseline Cross-Sheet Cosine Map (step={step})")
    plt.show()

## 7. Baseline threshold masks

In [ ]:
for step in save_steps:
    snap = snapshots_base[step]
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        snap["mask"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Baseline Below-Threshold Region (step={step})")
    plt.show()

## 8. Baseline central-band masks

In [ ]:
for step in save_steps:
    snap = snapshots_base[step]
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        snap["central"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Baseline Central Failed Region (step={step})")
    plt.show()

## 9. Baseline width and centerline snapshots

In [ ]:
for step in save_steps:
    snap = snapshots_base[step]
    plt.figure(figsize=(9, 4))
    plt.plot(x, snap["widths"])
    plt.xlabel("x")
    plt.ylabel("failed-band width in y")
    plt.title(f"Baseline Failed-Band Width vs x (step={step})")
    plt.show()

In [ ]:
for step in save_steps:
    snap = snapshots_base[step]
    plt.figure(figsize=(9, 4))
    plt.plot(x, snap["centerline"])
    plt.axhline(THRESHOLD, linestyle="--")
    plt.xlabel("x")
    plt.ylabel("cross-sheet cosine at y≈0")
    plt.title(f"Baseline Centerline Cosine (step={step})")
    plt.show()

## 10. Baseline time-series metrics

In [ ]:
steps = np.array(metrics_base["step"])

plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["component_count"])
plt.xlabel("step")
plt.ylabel("central component count")
plt.title("Baseline Pocket Count vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["width_std"])
plt.xlabel("step")
plt.ylabel("width std along x")
plt.title("Baseline Width Variation vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["centerline_min"])
plt.xlabel("step")
plt.ylabel("minimum centerline cosine")
plt.title("Baseline Deepest Centerline Shear vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["centerline_min_count"])
plt.xlabel("step")
plt.ylabel("centerline local minima count")
plt.title("Baseline Centerline Pocket Count Proxy vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["failed_fraction"], label="domain-wide below-threshold fraction")
plt.plot(steps, metrics_base["central_failed_fraction"], label="central-band below-threshold fraction")
plt.xlabel("step")
plt.ylabel("below-threshold fraction")
plt.title("Baseline Below-Threshold Fraction vs Time")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["constraint_energy"])
plt.xlabel("step")
plt.ylabel("constraint energy")
plt.title("Baseline Constraint Energy vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["corr_length_proxy"])
plt.xlabel("step")
plt.ylabel("correlation-length proxy")
plt.title("Baseline Correlation-Length Proxy vs Time")
plt.show()

## 11. Critical-step proxy

We use the largest growth rate in the below-threshold fraction as a simple proxy for a transition-like step.

In [ ]:
critical_step = metrics_base["critical_step_proxy"]
growth = np.gradient(metrics_base["failed_fraction"])

plt.figure(figsize=(8, 5))
plt.plot(steps, growth)
plt.axvline(critical_step)
plt.xlabel("step")
plt.ylabel("d/dt below-threshold fraction (discrete proxy)")
plt.title("Baseline Growth-Rate Proxy")
plt.show()

print("critical-step proxy =", critical_step)

## 12. Regime comparison

Now compare the baseline with a slightly stronger activity-driven regime.

In [ ]:
stronger_params = {
    "dt": 0.0018,
    "nu": 0.0010,
    "alpha": 0.22,
    "beta": 1.15,
    "gamma": 0.15,
    "n_steps": 220,
}

snapshots_strong, metrics_strong = run_simulation(Bx0, By0, Bx_target, seed_pattern, stronger_params, save_steps)

print("stronger critical-step proxy =", metrics_strong["critical_step_proxy"])

In [ ]:
steps_strong = np.array(metrics_strong["step"])

plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["failed_fraction"], label="baseline")
plt.plot(steps_strong, metrics_strong["failed_fraction"], label="stronger activity")
plt.xlabel("step")
plt.ylabel("domain-wide below-threshold fraction")
plt.title("Regime Comparison: Below-Threshold Fraction")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["constraint_energy"], label="baseline")
plt.plot(steps_strong, metrics_strong["constraint_energy"], label="stronger activity")
plt.xlabel("step")
plt.ylabel("constraint energy")
plt.title("Regime Comparison: Constraint Energy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["width_std"], label="baseline")
plt.plot(steps_strong, metrics_strong["width_std"], label="stronger activity")
plt.xlabel("step")
plt.ylabel("width std along x")
plt.title("Regime Comparison: Width Variation")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, metrics_base["component_count"], label="baseline")
plt.plot(steps_strong, metrics_strong["component_count"], label="stronger activity")
plt.xlabel("step")
plt.ylabel("central component count")
plt.title("Regime Comparison: Pocket Count")
plt.legend()
plt.show()

## 13. Compact snapshot summary

In [ ]:
for step in save_steps:
    idx = metrics_base["step"].index(step)
    print(
        f"baseline step={step:3d} | "
        f"failed_fraction={metrics_base['failed_fraction'][idx]:.4f} | "
        f"component_count={metrics_base['component_count'][idx]} | "
        f"width_std={metrics_base['width_std'][idx]:.4f} | "
        f"constraint_energy={metrics_base['constraint_energy'][idx]:.4f}"
    )

In [ ]:
for step in save_steps:
    idx = metrics_strong["step"].index(step)
    print(
        f"stronger step={step:3d} | "
        f"failed_fraction={metrics_strong['failed_fraction'][idx]:.4f} | "
        f"component_count={metrics_strong['component_count'][idx]} | "
        f"width_std={metrics_strong['width_std'][idx]:.4f} | "
        f"constraint_energy={metrics_strong['constraint_energy'][idx]:.4f}"
    )

## 14. Interpretation

This upgraded Notebook 04 supports a more complete statement:

> Fragmented current-sheet pockets can persist under pseudo-time dynamics while width variation, threshold violation, and constraint energy evolve in a bounded way.

What is improved relative to the earlier version:

- the dynamics are now tracked with more than one scalar metric
- the 45° gate is promoted to an explicit **constraint-energy order parameter**
- a critical-step proxy is extracted from the growth of threshold violation
- two parameter regimes can be compared directly

What this still does **not** claim:

- full plasma transport physics
- quantitative solar reconnection rates
- a replacement for MHD or PIC simulation

Best use:

> a geometric and dynamical precursor model for fragmented current sheets under constraint relaxation.